# Laboratorio 2: Modelado y Validación - AlpesPlanck

**Caso:** AlpesPlanck, estación meteorológica de Jena.

**Integrantes:** Juan Camilo Panadero - Jose Luis Parra

**Objetivo:** Desarrollar y evaluar modelos predictivos (Regresión Polinomial, Ridge y Lasso) para el problema planteado, optimizando hiperparámetros y cuantificando la incertidumbre.

In [22]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn: preprocesamiento y modelos
from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [27]:
# Cargar el conjunto etiquetado de entrenamiento del Laboratorio 1
# El archivo de prueba del Laboratorio 1 se reserva para predicciones finales.
df = pd.read_csv('../L1/data/Datos Lab 1.csv')

# Conservar solo filas con etiqueta y corregir los valores identificados en el Lab 1
# como temperaturas registradas en Fahrenheit.
df = df.dropna(subset=['temp_max_manana']).copy()
fahrenheit_mask = df['temp_max_manana'] > 35
df.loc[fahrenheit_mask, 'temp_max_manana'] = (
    df.loc[fahrenheit_mask, 'temp_max_manana'] - 32
) * 5 / 9

# Marcar valores fisicamente imposibles como faltantes para que el pipeline los impute.
valid_ranges = {
    'presion_media': (900, 1100),
    'presion_min': (900, 1100),
    'presion_max': (900, 1100),
    'humedad_media': (0, 100),
    'humedad_min': (0, 100),
    'humedad_max': (0, 100),
    'viento_media': (0, 50),
    'viento_min': (0, 50),
    'viento_max': (0, 50),
    'rafaga_media': (0, 50),
    'rafaga_min': (0, 50),
    'rafaga_max': (0, 50),
    'viento_norte': (-50, 50),
    'viento_este': (-50, 50),
    'direccion_viento': (0, 360),
    'registros_del_dia': (1, 400),
    'dia_del_anio': (1, 366)
}
invalid_values = 0
for column, (lower, upper) in valid_ranges.items():
    invalid_mask = ~df[column].between(lower, upper) & df[column].notna()
    invalid_values += int(invalid_mask.sum())
    df.loc[invalid_mask, column] = np.nan

# Separar variables predictoras y variable objetivo
X = df.drop(columns=['temp_max_manana', 'fecha'])
y = df['temp_max_manana']

categorical_features = ['estacion_anio', 'mes', 'sector_viento']
numeric_features = X.columns.difference(categorical_features).tolist()

# Division reproducible de los datos etiquetados
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f'Tamaño de entrenamiento: {X_train.shape}')
print(f'Tamaño de prueba: {X_test.shape}')
print(f'Valores faltantes en X: {int(X.isna().sum().sum())}')
print(f'Valores corregidos de Fahrenheit: {int(fahrenheit_mask.sum())}')
print(f'Valores imposibles convertidos a faltantes: {invalid_values}')

Tamaño de entrenamiento: (1860, 25)
Tamaño de prueba: (621, 25)
Valores faltantes en X: 1892
Valores corregidos de Fahrenheit: 15
Valores imposibles convertidos a faltantes: 29


## 1. Regresión Polinomial y Búsqueda de Hiperparámetros
En esta sección construimos un pipeline que incluye la generación de características polinomiales, escalamiento y un modelo de regresión lineal. Usaremos `GridSearchCV` para encontrar el grado óptimo.

In [28]:
# El polinomio se aplica solo a variables numéricas para evitar una expansión
# combinatoria de las categorías codificadas.
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features)
])

pipe_poly = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

param_grid_poly = {
    'preprocessor__numeric__poly__degree': [1, 2, 3],
    'preprocessor__numeric__scaler': [StandardScaler(), MinMaxScaler()]
}

grid_poly = GridSearchCV(
    estimator=pipe_poly,
    param_grid=param_grid_poly,
    cv=5,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    n_jobs=-1
)
grid_poly.fit(X_train, y_train)

print(f'Mejor configuración: {grid_poly.best_params_}')
print(f'RMSE promedio CV: {-grid_poly.best_score_:.4f}')

Mejor configuración: {'preprocessor__numeric__poly__degree': 1, 'preprocessor__numeric__scaler': StandardScaler()}
RMSE promedio CV: 6.8773


In [29]:
# Evaluación del mejor modelo polinomial sobre el conjunto de prueba etiquetado
best_poly = grid_poly.best_estimator_
y_pred_poly = best_poly.predict(X_test)

metrics_poly = pd.DataFrame({
    'Métrica': ['RMSE', 'MAE', 'R2'],
    'Valor': [
        np.sqrt(mean_squared_error(y_test, y_pred_poly)),
        mean_absolute_error(y_test, y_pred_poly),
        r2_score(y_test, y_pred_poly)
    ]
})

print(f'Rango real: {y_test.min():.2f} a {y_test.max():.2f}')
print(f'Rango predicho: {y_pred_poly.min():.2f} a {y_pred_poly.max():.2f}')
metrics_poly

Rango real: -9.71 a 34.38
Rango predicho: -83.56 a 29.61


,Métrica,Valor
0,RMSE,7.646574
1,MAE,5.158068
2,R2,0.243651
